In [0]:
from pyspark.sql.functions import col, when, expr

# 1. Cargar datos de la capa Silver
df_silver = spark.read.table("workspace.silver.framingham_clean")

# 2. Feature Engineering: Creación de variables con valor clínico
# En lugar de solo datos crudos, creamos indicadores que ayudan al modelo de ML
df_gold = df_silver.withColumn(
    # Categorización de Presión Arterial (Regla de negocio)
    "bp_category", 
    when((col("sysBP") < 120) & (col("diaBP") < 80), "Normal")
    .when((col("sysBP") < 130) & (col("diaBP") < 80), "Elevated")
    .otherwise("Hypertension")
).withColumn(
    # Índice de Riesgo: Pulse Pressure (Sistólica - Diastólica)
    "pulse_pressure", col("sysBP") - col("diaBP")
).withColumn(
    # Simplificación de educación (¿Es profesional o no?)
    "is_highly_educated", when(col("education") >= 3, 1).otherwise(0)
).withColumn(
    # Variable de riesgo por edad y cigarrillos
    "age_smoke_index", col("age") * col("cigsPerDay")
)

# 3. Selección final y casteo para el modelo
# Eliminamos columnas de auditoría de Bronze/Silver y dejamos solo lo necesario
final_features = [
    "male", "age", "is_highly_educated", "currentSmoker", "cigsPerDay", 
    "BPMeds", "prevalentStroke", "prevalentHyp", "diabetes", 
    "totChol", "sysBP", "diaBP", "BMI", "heartRate", "glucose",
    "bp_category", "pulse_pressure", "age_smoke_index",
    "TenYearCHD" # Target
]

df_gold_final = df_gold.select(*final_features)

# 4. Escritura optimizada (Liquid Clustering)
# Usamos Liquid Clustering en lugar de particionamiento tradicional para mejor performance
(df_gold_final.write
    .format("delta")
    .mode("overwrite")
    .option("clusterBy", "TenYearCHD, age") # Optimiza consultas filtradas por estas columnas
    .saveAsTable("workspace.gold.risk_prediction_features"))

print("Capa Gold creada y optimizada.")